# 🎯 SAM LAB — 프롬프트 세그멘테이션 3부작 (SAM 1 → 2 → 3)

> **[26년 3기] NPU 활용 온디바이스 AI 프로그래밍** · 특별 세션 (NPU 불필요 · Colab T4 GPU)

---

## 🔑 핵심 메시지

> **"무거운 인코더는 단 한 번, 가벼운 디코더는 무한히."**
>
> SAM이 클릭 한 번에 실시간으로 마스크를 그려낼 수 있는 이유는
> **이미지 임베딩을 1회만 계산하고 캐시**한 뒤, 프롬프트마다 초경량 디코더만 다시 돌리기 때문입니다.
> 이 비대칭 설계는 온디바이스 AI의 핵심 질문 — *"무엇을 무겁게, 무엇을 가볍게 만들 것인가"* — 와 정확히 같은 질문입니다.

## 📋 실습 로드맵

| Part | 모델 | 프롬프트 | 핵심 개념 |
|---|---|---|---|
| 1 | **SAM 1** (2023) | 점 · 박스 | 인코더/디코더 비대칭, 모호성(multimask) |
| 2 | **SAM 2** (2024) | 첫 프레임 클릭 | 비디오 전파, Memory Attention |
| 3 | **SAM 3** (2025) | **텍스트** | 개념(Concept) 세그멘테이션, Presence |
| 4 | 리포트 과제 | — | 실험 3종 |

## ⚙️ 실행 환경

- **런타임 → 런타임 유형 변경 → T4 GPU** 필수
- 전체 실행 시간: 📊 약 25~35분 (체크포인트 다운로드 포함)
- ⚠️ 본 노트북은 GPU 비결정성 환경이므로 시간 측정값은 📊(기대 범위)로 표기합니다.


---
# Part 0 · 환경 설정

세 세대의 SAM을 한 노트북에서 돌리기 위한 공통 환경을 구성합니다.

> 🧑‍🏫 **강사 노트**: 아래 설치 셀은 SAM 1/2가 서로 다른 패키지(`segment-anything`, `sam2`)로 배포되기 때문에 둘 다 설치합니다. 충돌은 없습니다. SAM 3은 Part 3에서 별도 설치합니다.

In [ ]:
# [0-1] GPU 확인 — "Tesla T4"가 보여야 합니다
!nvidia-smi -L

**📊 기대 출력**
```
GPU 0: Tesla T4 (UUID: GPU-xxxxxxxx-....)
```
> ⚠️ `command not found`가 나오면 런타임 유형이 CPU입니다. 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 다시 실행하세요.

In [ ]:
# [0-2] SAM 1 · SAM 2 설치 (약 1~2분)
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install -q 'git+https://github.com/facebookresearch/sam2.git'
!pip install -q supervision opencv-python matplotlib
print("✅ 설치 완료")

In [ ]:
# [0-3] 체크포인트 다운로드
# 💡 수업 원칙: urllib 대신 curl -sL 사용 (리다이렉트 안전)
import os
os.makedirs("checkpoints", exist_ok=True)

# SAM 1 — ViT-B (가장 작은 백본, 375MB)
!curl -sL -o checkpoints/sam_vit_b.pth \
    https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

# SAM 2.1 — Hiera-Small (비디오용, 184MB)
!curl -sL -o checkpoints/sam2.1_hiera_small.pt \
    https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

!ls -lh checkpoints/

**📊 기대 출력** — 파일 2개가 각각 약 `358M`, `176M` 크기로 보이면 성공입니다.

> 🧑‍🏫 **강사 노트**: 다운로드가 0바이트로 끝나는 학생이 있으면 Meta CDN 일시 장애입니다. `curl` 재실행으로 대부분 해결되며, 안 되면 미러 링크(수업 전 강사가 Google Drive에 준비)를 안내하세요.

In [ ]:
# [0-4] 공통 유틸리티 — 시각화 함수
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2

np.random.seed(42)  # 마스크 색상 재현용

def show_mask(mask, ax, random_color=False, alpha=0.55):
    """마스크를 반투명 컬러로 오버레이"""
    if random_color:
        color = np.concatenate([np.random.random(3), [alpha]])
    else:
        color = np.array([30/255, 144/255, 255/255, alpha])  # 파란색
    h, w = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * color.reshape(1, 1, -1))

def show_points(coords, labels, ax, size=300):
    """전경(★초록)/배경(★빨강) 프롬프트 점 표시"""
    pos = coords[labels == 1]; neg = coords[labels == 0]
    ax.scatter(pos[:, 0], pos[:, 1], color='lime', marker='*',
               s=size, edgecolor='white', linewidth=1.5)
    ax.scatter(neg[:, 0], neg[:, 1], color='red', marker='*',
               s=size, edgecolor='white', linewidth=1.5)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h,
                 edgecolor='lime', facecolor=(0,0,0,0), lw=2.5))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ device = {device}")

In [ ]:
# [0-5] 실습 이미지 다운로드
# 과정 연계: African Wildlife와 같은 야생동물 도메인의 이미지를 사용합니다
os.makedirs("images", exist_ok=True)
!curl -sL -o images/wildlife.jpg \
    https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/truck.jpg

image_bgr = cv2.imread("images/wildlife.jpg")
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print(f"이미지 크기: {image.shape}")   # 📊 (1200, 1800, 3)

plt.figure(figsize=(8, 5)); plt.imshow(image); plt.axis('off')
plt.title("실습 이미지"); plt.show()

> 🧑‍🏫 **강사 노트**: 위 셀은 SAM 공식 예제 이미지(truck.jpg)를 기본으로 사용합니다. **수업 전 준비**: 우리 과정의 African Wildlife 데이터셋 이미지 1장(코끼리 무리 등, 인스턴스가 여러 개인 것)을 Drive 공유 링크로 준비해서 교체하면 Part 3의 "개념 세그멘테이션"에서 서사가 완성됩니다. 어떤 이미지를 쓰든 노트북은 동일하게 동작합니다.

---
# Part 1 · SAM 1 — 점 하나로 무엇이든 분리하기

## 1-0. 구조 먼저 이해하기

```
                       (무겁다 · 1회)                    (가볍다 · 프롬프트마다)
 이미지 ──▶ ┌──────────────────┐   임베딩    ┌─────────────────┐
            │  Image Encoder    │──▶ 캐시 ──▶│  Mask Decoder    │──▶ 마스크 3장 + 점수
            │  ViT-B  (~91M)    │  (64×64)   │  (~4M, 2-layer)  │
            └──────────────────┘             └────────┬────────┘
                                                      ▲
                                       점·박스 ──▶ Prompt Encoder (초경량)
```

- **Image Encoder**: 전체 파라미터의 약 95%. 이미지당 **딱 1번** 실행
- **Mask Decoder**: 약 4M 파라미터. 프롬프트가 바뀔 때마다 **이것만** 재실행
- 이 비대칭이 오늘 실습의 클라이맥스입니다 (1-5에서 실측)

In [ ]:
# [1-1] SAM 1 로드
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_b"](checkpoint="checkpoints/sam_vit_b.pth")
sam.to(device)
predictor = SamPredictor(sam)

n_total = sum(p.numel() for p in sam.parameters())
n_enc   = sum(p.numel() for p in sam.image_encoder.parameters())
n_dec   = sum(p.numel() for p in sam.mask_decoder.parameters())
print(f"전체 파라미터   : {n_total/1e6:6.1f} M")
print(f"이미지 인코더   : {n_enc/1e6:6.1f} M  ({100*n_enc/n_total:.1f}%)")
print(f"마스크 디코더   : {n_dec/1e6:6.1f} M  ({100*n_dec/n_total:.1f}%)")

**✅ 기대 출력** (파라미터 수는 결정적 — 고정값)
```
전체 파라미터   :   93.7 M
이미지 인코더   :   89.7 M  (95.7%)
마스크 디코더   :    4.1 M  (4.3%)
```

숫자가 말해줍니다 — **인코더가 전체의 95.7%** 입니다. 그런데도 SAM이 "실시간 클릭"이 가능한 이유를 지금부터 확인합니다.

In [ ]:
# [1-2] 이미지 임베딩 계산 — 무거운 작업, 그러나 단 1회
import time

torch.cuda.synchronize()
t0 = time.perf_counter()
predictor.set_image(image)          # ← 인코더 실행은 여기서 딱 1번
torch.cuda.synchronize()
t_encode = (time.perf_counter() - t0) * 1000

emb = predictor.get_image_embedding()
print(f"임베딩 shape : {tuple(emb.shape)}")      # ✅ (1, 256, 64, 64)
print(f"인코딩 시간  : {t_encode:.0f} ms  📊 (T4 기준 대략 300~900ms)")

> 💡 원본 이미지 1200×1800×3 (약 650만 값) → 임베딩 1×256×64×64 (약 105만 값).
> 이미지의 "의미"가 이 텐서 안에 압축되었고, 이제부터 프롬프트는 이 캐시만 참조합니다.

In [ ]:
# [1-3] 점 프롬프트 — 클릭 한 번으로 마스크 얻기
input_point = np.array([[500, 375]])   # (x, y) — 객체 위의 한 점
input_label = np.array([1])            # 1=전경(이걸 원해), 0=배경(이건 빼줘)

masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=False,
)
print(f"마스크 shape: {masks.shape}, 예측 IoU 점수: {scores[0]:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(image); show_mask(masks[0], ax); show_points(input_point, input_label, ax)
ax.set_title(f"점 1개 → 마스크 (score {scores[0]:.3f})"); ax.axis('off'); plt.show()

In [ ]:
# [1-4] 같은 점, 세 가지 답 — 모호성(Ambiguity) 실험
# 한 점은 "부분/전체/부품" 어느 것이든 가리킬 수 있습니다.
# multimask_output=True 는 세 후보를 모두 돌려줍니다.
masks3, scores3, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,        # ← 유일한 변경점
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, ax in enumerate(axes):
    ax.imshow(image); show_mask(masks3[i], ax)
    show_points(input_point, input_label, ax, size=200)
    ax.set_title(f"후보 {i+1} · score {scores3[i]:.3f} · 면적 {masks3[i].sum():,}px")
    ax.axis('off')
plt.tight_layout(); plt.show()

print("📊 관찰 포인트: 세 마스크의 면적이 크게 다릅니다 (부품 → 부분 → 전체 위계)")

### 🤔 생각해 볼 것

| 질문 | 답 |
|---|---|
| 왜 후보가 하필 **3개**인가? | 학습 시 "부품(subpart)-부분(part)-전체(whole)" 3계층 모호성을 커버하도록 설계 |
| score는 무엇인가? | 모델이 **스스로 예측한 IoU** (실제 GT와의 IoU가 아님 — 자기평가값) |
| 실전에서는 어느 걸 쓰나? | 보통 `scores.argmax()` — 하지만 리포트 실험 ①에서 이 전략이 항상 옳은지 검증합니다 |

In [ ]:
# [1-5] 🎬 클라이맥스 — 인코더 1회 vs 디코더 N회 실측
# 프롬프트 50개를 연속으로 던져서, 임베딩 캐시 덕분에
# 디코더가 얼마나 가벼운지 숫자로 확인합니다.

h, w = image.shape[:2]
rng = np.random.default_rng(42)                    # seeded — 점 위치 재현 가능
pts = rng.integers([0, 0], [w, h], size=(50, 2))

torch.cuda.synchronize()
t0 = time.perf_counter()
for p in pts:
    predictor.predict(point_coords=p[None, :],
                      point_labels=np.array([1]),
                      multimask_output=False)
torch.cuda.synchronize()
t_decode_each = (time.perf_counter() - t0) * 1000 / 50

print("┌──────────────────────────────────────────────────┐")
print(f"│ 인코더 (1회)        : {t_encode:7.0f} ms               │")
print(f"│ 디코더 (프롬프트당) : {t_decode_each:7.1f} ms               │")
print(f"│ 비율                : 약 {t_encode/t_decode_each:5.0f} : 1             │")
print("└──────────────────────────────────────────────────┘")
print("📊 기대: 인코더 300~900ms vs 디코더 5~20ms → 수십~백 배 차이")

### 📌 Part 1 핵심 정리

> 클릭이 실시간인 이유 = **비싼 계산(인코더)을 캐시하고, 싼 계산(디코더)만 반복**하기 때문.
>
> 이 설계 원리는 우리 과정과 이렇게 연결됩니다:
> - **KD 실습에서 배운 것**: 무거운 부분을 작은 모델로 증류 → MobileSAM/EdgeSAM이 정확히 인코더를 증류한 사례
> - **NPU 배포 관점**: "인코더는 서버/1회, 디코더는 엣지/실시간" 같은 분할 배포 아키텍처가 가능해짐

---
# Part 2 · SAM 2 — 클릭 한 번이 비디오 전체로

SAM 1은 이미지 한 장에서 끝났습니다. SAM 2의 질문은:

> **"첫 프레임에서 클릭한 그 객체를, 나머지 모든 프레임에서 자동으로 따라갈 수 없을까?"**

핵심 장치는 **Memory Attention** — 과거 프레임의 (임베딩 + 마스크) 쌍을 **메모리 뱅크**에 쌓아두고,
새 프레임을 처리할 때 그 기억을 참조합니다.

```
프레임 t ─▶ 인코더 ─▶ [Memory Attention] ─▶ 디코더 ─▶ 마스크 t ─┐
                            ▲                                    │
                     메모리 뱅크 (최근 프레임들의 특징+마스크) ◀──┘  (기억으로 적립)
```

- Detection+Tracking(우리가 배운 ByteTrack)과 달리 **별도 tracker 없이 모델 내부 기억**으로 추적
- occlusion(가림) 후 재등장도 기억 덕분에 같은 객체로 이어집니다

In [ ]:
# [2-1] 샘플 비디오 준비 — 프레임 시퀀스로 분해
# SAM 2의 비디오 API는 "JPEG 프레임 폴더"를 입력으로 받습니다.
os.makedirs("video_frames", exist_ok=True)

!curl -sL -o bedroom.zip https://dl.fbaipublicfiles.com/segment_anything_2/assets/bedroom.zip
!unzip -oq bedroom.zip -d video_frames_src
# 프레임 폴더 정리 (앞쪽 60프레임만 사용해 시간 절약)
import glob, shutil
frames = sorted(glob.glob("video_frames_src/bedroom/*.jpg"))[:60]
for i, fp in enumerate(frames):
    shutil.copy(fp, f"video_frames/{i:05d}.jpg")
print(f"✅ 프레임 {len(frames)}장 준비 완료")

> 🧑‍🏫 **강사 노트**: 공식 데모 영상(bedroom, 아이가 방에서 움직이는 장면)을 사용합니다. **수업 전 준비**: 보드 실습 때 찍어둔 교실/사무실 짧은 영상(5초 내외)을 `ffmpeg -i input.mp4 -q:v 2 video_frames/%05d.jpg`로 변환해서 쓰면 학생 몰입도가 훨씬 좋습니다. zip 다운로드 실패 시(📊 드물게 CDN 오류) 이 ffmpeg 경로가 대안입니다.

In [ ]:
# [2-2] SAM 2 비디오 predictor 로드 + 전체 프레임 사전 인코딩
from sam2.build_sam import build_sam2_video_predictor

predictor2 = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_s.yaml",
    "checkpoints/sam2.1_hiera_small.pt",
    device=device,
)

t0 = time.perf_counter()
state = predictor2.init_state(video_path="video_frames")   # 전 프레임 인코딩
t_init = time.perf_counter() - t0
print(f"✅ init_state 완료 — {t_init:.1f}s  📊 (60프레임, T4 기준 대략 10~30s)")
print("   ↳ SAM 1과 같은 원리: 인코딩은 미리 다 해두고, 프롬프트/전파는 그 캐시 위에서")

In [ ]:
# [2-3] 첫 프레임에 클릭 1번 → 전체 비디오로 전파
ann_frame_idx = 0    # 첫 프레임에
ann_obj_id    = 1    # 객체 ID 1번으로

# 📊 bedroom 영상 기준 아이(인물) 위의 점. 다른 영상이면 좌표를 바꿔주세요.
click_point = np.array([[210, 350]], dtype=np.float32)
click_label = np.array([1], np.int32)

_, obj_ids, mask_logits = predictor2.add_new_points_or_box(
    inference_state=state, frame_idx=ann_frame_idx,
    obj_id=ann_obj_id, points=click_point, labels=click_label,
)

# --- 전파(propagate): 클릭 없이 나머지 프레임 자동 추적 ---
video_segments = {}
t0 = time.perf_counter()
for f_idx, ids, logits in predictor2.propagate_in_video(state):
    video_segments[f_idx] = (logits[0] > 0.0).cpu().numpy()
t_prop = time.perf_counter() - t0

print(f"✅ {len(video_segments)}프레임 전파 완료 — {t_prop:.1f}s "
      f"(프레임당 📊 {1000*t_prop/len(video_segments):.0f}ms)")

In [ ]:
# [2-4] 결과 확인 — 4개 시점 샘플링
sample_idx = [0, 19, 39, 59]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, fi in zip(axes, sample_idx):
    frame = cv2.cvtColor(cv2.imread(f"video_frames/{fi:05d}.jpg"), cv2.COLOR_BGR2RGB)
    ax.imshow(frame)
    if fi in video_segments:
        show_mask(video_segments[fi][0], ax)
    ax.set_title(f"frame {fi}" + ("  ← 클릭은 여기 1번뿐" if fi == 0 else ""))
    ax.axis('off')
plt.suptitle("클릭 1번 → 60프레임 자동 전파", y=1.03); plt.tight_layout(); plt.show()

### 📌 Part 2 관찰 포인트

- **클릭은 frame 0에 단 1번** — 나머지 59프레임은 메모리 뱅크가 이어갔습니다
- 📊 중간에 객체가 잠시 가려지는 구간이 있다면: 마스크가 사라졌다가 재등장 시 **같은 객체로 복귀**하는지 확인하세요 (ByteTrack의 2-pass 매칭과 같은 문제를 전혀 다른 방식으로 푼 것)
- 결과가 도중에 다른 객체로 "새는" 학생이 있다면: 그 프레임에서 배경 점(`label=0`)을 추가 클릭해 교정하는 것이 SAM 2의 공식 워크플로우입니다 (교정 후 재전파)

> 🧑‍🏫 **강사 노트 (양방향)**: bedroom 영상에서는 60프레임 내 심한 occlusion이 **없을 수도 있습니다**. 안 나타나면 "이 영상은 순한 케이스"라고 언급하고 리포트 실험 ②(직접 촬영 영상에서 가림 실험)로 넘기면 됩니다. 나타나면 그 자리에서 프레임을 앞뒤로 넘겨 보여주세요 — 그게 그날의 클라이맥스가 됩니다.

---
# Part 3 · SAM 3 — 클릭 대신 "말"로 시키기

SAM 1/2의 프롬프트는 **기하학적**(점·박스)이었습니다. 인스턴스 **하나**를 가리키는 방식이죠.

SAM 3 (2025.11 공개)의 프롬프트는 **개념(concept)** 입니다:

> `"yellow fish"` 라고 말하면 → 이미지/비디오 안의 **모든 노란 물고기**를 각각 분리

| | SAM 1/2 | SAM 3 |
|---|---|---|
| 프롬프트 | 점, 박스 | **명사구 텍스트**, 예시 이미지(exemplar) |
| 대상 | 가리킨 인스턴스 1개 | 개념에 해당하는 **모든 인스턴스** |
| 새 장치 | — | **Presence Head**: "이 개념이 이미지에 존재하는가?"를 먼저 판정 |

Presence Head가 중요한 이유: 텍스트로 물으면 모델은 "없는 것도 억지로 찾으려는" 유혹에 빠집니다.
존재 여부 판정을 분리하면 **없을 때 깔끔하게 빈 결과**를 낼 수 있습니다 (오탐 억제 — 우리 스마트 안전 트랙의 핵심 과제와 동일한 문제의식).

In [ ]:
# [3-1] SAM 3 설치 — Hugging Face transformers 경유
# ⚠️ SAM 3은 HF 게이트 모델일 수 있습니다. 최초 1회:
#   1) https://huggingface.co/facebook/sam3 에서 약관 동의
#   2) 아래 로그인 셀에서 HF 토큰 입력 (read 권한이면 충분)
!pip install -q -U transformers accelerate

from huggingface_hub import notebook_login
notebook_login()   # 토큰 입력 UI가 뜹니다

> 🧑‍🏫 **강사 노트 (중요)**: 이 Part는 외부 API 의존도가 가장 높습니다. **수업 전 반드시 최신 transformers에서 아래 셀 실행 확인** 후, 정상 동작한 버전을 `pip install transformers==X.Y.Z`로 고정해 이 셀에 명시하세요 (TACHY-Compiler 0.1.0 고정과 같은 원칙). HF 접속 장애 대비: 강사 계정으로 미리 받은 모델을 Drive에 캐시(`~/.cache/huggingface` 통째 복사)해 두면 오프라인 복구가 가능합니다.

In [ ]:
# [3-2] SAM 3 로드 + 텍스트 프롬프트 개념 세그멘테이션
from transformers import Sam3Processor, Sam3Model
from PIL import Image as PILImage

processor = Sam3Processor.from_pretrained("facebook/sam3")
model3 = Sam3Model.from_pretrained("facebook/sam3", torch_dtype=torch.float16).to(device)

pil_img = PILImage.fromarray(image)
TEXT = "wheel"        # 🔁 African Wildlife 이미지로 바꿨다면 "elephant" 등으로

inputs = processor(images=pil_img, text=TEXT, return_tensors="pt").to(device)
with torch.no_grad():
    out = model3(**inputs)

res = processor.post_process_instance_segmentation(
    out, threshold=0.5, mask_threshold=0.5,
    target_sizes=[pil_img.size[::-1]])[0]

print(f'"{TEXT}" → 인스턴스 {len(res["masks"])}개 검출  📊 (truck 이미지 + "wheel" 기준 2~3개)')

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(image)
for m in res["masks"]:
    show_mask(m.cpu().numpy(), ax, random_color=True)
ax.set_title(f'텍스트 프롬프트: "{TEXT}" — 해당하는 모든 인스턴스'); ax.axis('off'); plt.show()

In [ ]:
# [3-3] Presence 실험 — 없는 개념을 물어보면?
# 오탐 억제 능력을 직접 시험합니다.
for text in [TEXT, "zebra", "penguin"]:      # 뒤 2개는 이 이미지에 없음
    inputs = processor(images=pil_img, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model3(**inputs)
    res = processor.post_process_instance_segmentation(
        out, threshold=0.5, mask_threshold=0.5,
        target_sizes=[pil_img.size[::-1]])[0]
    print(f'  "{text:10s}" → {len(res["masks"])}개')

print()
print("📊 기대: 존재하는 개념만 0이 아닌 개수, 없는 개념은 0개")
print("   (0이 아니게 나오면 그 자체가 훌륭한 토론 소재 — threshold를 올려보세요)")

### 📌 Part 3 핵심 정리 + 세 세대 총정리

| | 프롬프트 | 출력 | 한 문장 요약 |
|---|---|---|---|
| **SAM 1** | 점·박스 | 마스크 1개(×3후보) | "여기, 이거 분리해줘" |
| **SAM 2** | 첫 프레임 점 | 비디오 전체 마스크 | "얘를 계속 따라가줘" |
| **SAM 3** | 텍스트 개념 | 모든 인스턴스 마스크 | "**이런 것들** 전부 찾아줘" |

세 세대를 관통하는 설계는 변하지 않았습니다:
**무거운 인코더(1회) + 캐시 + 가벼운 프롬프트 처리(N회)** — 프롬프트의 종류만 진화했습니다.

---
# Part 4 · 리포트 과제 (3종 중 2종 선택)

### 실험 ① — "최고 점수 마스크가 항상 정답인가?"
`[1-4]`를 이미지 내 **10군데** 다른 위치에서 반복하고(seeded 점 목록 제공됨: `rng = np.random.default_rng(7)`),
`scores.argmax()`가 고른 마스크가 **당신이 의도한** 마스크와 일치한 비율을 기록하세요.
- 예상을 깨는 지점: 객체 경계 근처 점에서는 argmax가 자주 "전체"가 아닌 "부품"을 고릅니다. 왜일까요?

### 실험 ② — 가림(occlusion) 스트레스 테스트
휴대폰으로 5초 영상을 찍되, **객체가 2초간 완전히 가려지는 장면**을 연출하세요 (손으로 컵 가리기 등).
SAM 2로 전파한 뒤: 재등장 시 같은 객체로 복귀하는가? 몇 프레임 만에? 실패한다면 어떤 조건(가림 시간, 배경 유사도)에서?

### 실험 ③ — 프롬프트 문구 민감도 (SAM 3)
같은 대상에 대해 표현을 5가지로 바꿔보세요 (예: `"wheel"` / `"tire"` / `"black wheel"` / `"front wheel"` / `"round object"`).
검출 개수와 마스크 품질이 어떻게 달라지는지 표로 정리하세요.
- 연결: CLIP 계열 텍스트 이해의 특성 + "프롬프트 엔지니어링"이 왜 생겼는지에 대한 실증

---

## ✅ 체크포인트 — 오늘 확보해야 할 3가지

1. **비대칭 실측표** `[1-5]` — 인코더:디코더 시간 비율 스크린샷
2. **전파 결과 4분할** `[2-4]` — 클릭 1번 → 60프레임
3. **Presence 실험 출력** `[3-3]` — 없는 개념 0개 확인

> 🔗 **NPU 파이프라인과의 연결 (교육자 노트)**
> SAM의 ViT 인코더는 우리 BlackSwan NPU에 바로 올라가지 않습니다 (Transformer 연산 Fallback — Day 2에서 배운 그 문제).
> 그래서 실무에서는 ① 인코더를 CNN으로 **지식 증류**(MobileSAM/EdgeSAM — 우리 KD 실습의 실전판)하거나
> ② 인코더는 서버에, 디코더만 엣지에 두는 **분할 배포**를 씁니다.
> "무엇을 무겁게, 무엇을 가볍게" — SAM 설계자와 온디바이스 엔지니어는 같은 질문에 답하고 있습니다.